In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / 'core').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3, 'figure.figsize': (7, 4)})
FIG = ROOT / 'results' / 'figures'; DATA = ROOT / 'results' / 'data'

# 06 — Side note: substitutes ($\varepsilon<0$)

The proposal fixes $\varepsilon\ge0$. Out of curiosity we let $\varepsilon<0$ (the goods are
substitutes: holding both is worth less than the sum). **Caveat:** for $\varepsilon<-v$ the
marginal value of item 1 becomes negative when the agent already holds item 2, and we do *not*
model free disposal, so the exercise is only meaningful for mild substitutability, say
$\varepsilon\in[-\tfrac12,0)$.

* First best: pooling now gives $1+\varepsilon<1$, so $\mathrm{OPT}=1+|x-y|$ — the additive split.
* The McCormick LP needs no change (utilities are still bilinear).
* The fixed-menu LIP mechanism is still strategy-proof; the dictator's objective is no longer
  guaranteed concave, so we use the robust grid+refine solver of `core.menu`.

In [2]:
from core import lp
from core.menu import LIPMenu, MenuMechanism
rows = []
for e in (-0.5, -0.25, -0.1, 0.0, 0.1, 0.25):
    r = lp.solve_sp_lp(30, eps=e, symmetric=True)
    best = max(((MenuMechanism(LIPMenu(a), e, nested=False, grid=801).grid_min(150)[0], a) for a in (1.0, 1.5, 2.0, 2.1, 2.5, 3.0, 4.0)))
    rows.append({"eps": e, "SP upper bound (LP, N=30)": r.alpha, "best eps-aware LIP (a in coarse set)": best[0], "a": best[1],
                 "equal split": (1 + e / 2) / 2, "take-all": (1 + e) / 2})
pd.DataFrame(rows).round(4)

,eps,"SP upper bound (LP, N=30)",best eps-aware LIP (a in coarse set),a,equal split,take-all
0,-0.50,0.7563,0.6667,4.0,0.3750,0.250
1,-0.25,0.8139,0.7596,3.0,0.4375,0.375
2,-0.10,0.8413,0.8028,2.5,0.4750,0.450
3,0.00,0.8454,0.8290,2.1,0.5000,0.500
4,0.10,0.8792,0.8454,1.5,0.5250,0.550
5,0.25,0.9166,0.8462,2.0,0.5625,0.625


With substitutes the picture is the mirror image of complementarities: the first best never
pools, the LP bound *falls* below the additive 0.84 (the designer cannot exploit any
"agreement" between agents, and the dictator's complementarity-driven over-buying now hurts),
and the deterministic $\varepsilon$-aware LIP keeps most of its performance — the menu class is
less handicapped because splitting is what menus do well. We leave a proper treatment (with
free disposal) as future work.